In [1]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q langchain langgraph
!pip install -q pydantic
!pip install -q google-generativeai timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.7 MB/s eta 0:00:00:00:0100:01


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import re,json

In [3]:
# MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.float16,
#     device_map="auto"
# )
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)
print("Model Loaded Successfully!")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model Loaded Successfully!


In [4]:
#All pydantic checks....

from pydantic import BaseModel
from typing import Optional
class MLContract(BaseModel):
    task_type: str
    dataset_domain: str
    target_metric: str
    target_value: float
    compute_constraint: str

class FeasibilityReport(BaseModel):
    feasible: bool
    reason: str
    suggested_target: float

class NegotiationResult(BaseModel):
    feasible_after_revision: bool
    revised_contract: dict
    optimization_suggestions: dict
    changes_made: list[str]
    reason: str

class TrainingPlan(BaseModel):
    recommended_model: str
    optimizer: str
    learning_rate: float
    batch_size: int
    epochs: int
    loss_function: str
    augmentations: list[str]
    training_strategies: list[str]
    resource_optimizations: list[str]


class ExecutionSpec(BaseModel):
    framework: str
    task_type: str
    dataset_understanding: dict
    model_spec: dict
    training_config: dict
    augmentation_config: list[str]
    optimization_config: list[str]
    evaluation_config: dict
    hardware_config: dict
    runtime_requirements: dict
    agent_notes: Optional[list[str]] = None


In [5]:
def contact_to_json(user_request,model):
    SYSTEM_PROMPT = """
    You are an Autonomous ML Contractor.
    
    Your task is to convert machine learning project requests into structured contracts.
    
    Extract the following fields:
    - task_type
    - dataset_domain
    - target_metric
    - target_value
    - compute_constraint
    
    Rules:
    1. Return ONLY valid JSON
    2. No explanations
    3. No markdown
    4. If missing, infer intelligently
    
    Example Output:
    {
        "task_type": "image_classification",
        "dataset_domain": "medical imaging",
        "target_metric": "accuracy",
        "target_value": 0.90,
        "compute_constraint": "8GB VRAM"
    }
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_request}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    # print(text)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False
    )
    
    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
    
    
    assistant_response = response.split("assistant")[-1].strip()
    # print(assistant_response)
    json_match = re.search(r"\{[\s\S]*?\}", assistant_response)
    
    contract_json = json.loads(json_match.group())
    
    #print(contract_json)
    try:
        contract = MLContract(**contract_json)
        return contract_json
    except:
        print("Pydantic check fail at contract_parsing")
        return -1


In [6]:

def safe_json_parse(text):
    try:
        json_match = re.search(
            r"\{[\s\S]*?\}",
            text
        )
        if json_match is None:
            return None
        json_text = json_match.group()
        return json.loads(json_text)
    except Exception as e:
        print("JSON Parsing Error:")
        print(e)
        return None

# FEASIBILITY AGENT
def feasibility_agent(contract, model):
    FEASIBILITY_SYSTEM_PROMPT = """
You are a Machine Learning Feasibility Analyzer.

Your task is to determine whether a machine learning contract is realistic.

You must consider:
- task complexity
- target metric
- compute limitations
- dataset domain

IMPORTANT RULES:
- Return ONLY valid JSON
- No markdown
- No explanations outside JSON
- suggested_target MUST be a FLOAT
- feasible MUST be true or false

Required JSON fields:
- feasible
- reason
- suggested_target

Example Output:

{
  "feasible": false,
  "reason": "Target accuracy is unrealistic under compute constraints.",
  "suggested_target": 0.90
}
"""
    contract_text = json.dumps(contract,indent=2)
    messages = [
        {
            "role": "system",
            "content": FEASIBILITY_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": contract_text
        }
    ]
    text = tokenizer.apply_chat_template(
        messages,tokenize=False,add_generation_prompt=True
    )
    inputs = tokenizer(text,return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        temperature=0.0,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
    assistant_response = response.split(
        "assistant"
    )[-1].strip()
    #print(assistant_response)
    feasibility_json = safe_json_parse(
        assistant_response
    )
    if feasibility_json is None:
        print("Failed JSON extraction")
        return -1, -1
    try:
        feasibility_json["suggested_target"] = float(feasibility_json["suggested_target"])

    except:
        print("Failed float conversion")
        return -1, -1
    try:

        report = FeasibilityReport(**feasibility_json)
        return feasibility_json, contract

    except Exception as e:
        print("\nPydantic Validation Failed")
        print(e)
        print("\nGenerated JSON:\n")
        print(feasibility_json)
        return -1, -1

In [7]:
# user_request = """
# Build a email spam classifier with around 98% accuracy under 6GB VRAM
# """
# contract=contact_to_json(user_request,model)
# print(contract)
# report=feasibility_agent(contract,model)
# print(report[0])


In [8]:
def negotiation_agent(contract,feasibility,model):
    NEGOTIATION_SYSTEM_PROMPT = """
You are an Autonomous ML Negotiation Agent.

Your task is to revise infeasible machine learning contracts while preserving user intent as much as possible.

You should NOT only reduce target metrics.

You should explore multiple optimization strategies including:
- lightweight architectures
- mixed precision training
- gradient accumulation
- transfer learning
- reduced image resolution
- smaller batch sizes
- compute upgrades
- reduced training duration

Your goal is to find the MINIMAL modifications needed to make the contract feasible.

Rules:
- Return ONLY valid JSON
- No explanations outside JSON

Required JSON fields:
- feasible_after_revision
- revised_contract
- optimization_suggestions
- changes_made
- reason
Always give optimization suggestion does not matter if revised contract is feasible or not.
Example:

Input Contract:
{
  "task_type": "image_classification",
  "target_metric": "accuracy",
  "target_value": 0.99,
  "compute_constraint": "2GB VRAM"
}
Feasibility Report:
{
  "feasible": false,
  "reason": "Target unrealistic under compute constraints"
}

Output:
{
  "feasible_after_revision": false,

  "revised_contract": {
    "task_type": "image_classification",
    "target_metric": "accuracy",
    "target_value": 0.95,
    "compute_constraint": "2GB VRAM"
  },

  "optimization_suggestions": {
    "recommended_model_family": "MobileNetV3",
    "mixed_precision": true,
    "gradient_accumulation": true,
    "reduced_image_resolution": "128x128",
    "suggested_compute_upgrade": "4GB VRAM"
  },

  "changes_made": [
    "Reduced target accuracy from 0.99 to 0.95",
    "Suggested lightweight architecture",
    "Enabled mixed precision"
  ],

  "reason": "Further optimization or slightly higher compute may be required"
}
"""
    negotiation_input = {
        "contract": contract,
        "feasibility_report": feasibility
    }
    negotiation_text = json.dumps(negotiation_input,indent=2)
    #print(negotiation_text)
    messages = [
        {
            "role": "system",
            "content": NEGOTIATION_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": negotiation_text
        }
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text,return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        temperature=0.0,
        repetition_penalty=1.1
    )
    response = tokenizer.decode(outputs[0],skip_special_tokens=True)
    #print(response)
    assistant_response = response.split("assistant")[-1].strip()
    #print(assistant_response)
    json_match = re.search(
        r"\{[\s\S]*\}",
        assistant_response
    )
    negotiation_json = json.loads(json_match.group())
    #print(negotiation_json)
    try:
        negotiation_result = NegotiationResult(**negotiation_json)
        return negotiation_json
    except:
        print("pynatic check fail ")
        return -1


In [9]:
def contract_distance(original, revised):

    score = 0
    if ("target_value" in original and "target_value" in revised):
        diff = abs(original["target_value"]- revised["target_value"])
        score += diff * 100

    # Compute change penalty
    if (original.get("compute_constraint")!= revised.get("compute_constraint")):
        score += 15
    return score

In [10]:
def autonomous_contract_optimizer(user_request,model,max_iterations=3):
    history = []
    original_contract = contact_to_json(user_request,model)
    if original_contract == -1:
        return {"status": "contract_parse_failed"}
    current_contract = original_contract
    for iteration in range(max_iterations):
        print(f"\n=== Iteration {iteration+1} ===")
        feasibility, _ = feasibility_agent(current_contract,model)
        if feasibility == -1:
            return {"status": "feasibility_failed"}
        print("Feasibility:", feasibility)

        history.append({
            "iteration": iteration + 1,
            "contract": current_contract,
            "feasibility": feasibility
        })
        
        negotiation = negotiation_agent(current_contract,feasibility,model)
        if negotiation == -1:
            return {"status": "negotiation_failed"}

        history[-1]["negotiation"] = negotiation
        optimization_suggestions = negotiation.get( "optimization_suggestions",{})
        #if feasible keep contract but also add optimization here that was an issue...
        if feasibility["feasible"]:
            print("\nFeasible contract found!")
            return {
                "status": "success",
                "final_contract": current_contract,
                "feasibility": feasibility,
                "optimization_suggestions":optimization_suggestions,
                "history": history
            }
        #if not feasible then revising the contract
        revised_contract = negotiation["revised_contract"]
        #print("Revised Contract:",revised_contract)
        
        distance = contract_distance(original_contract,revised_contract)
        #print("Contract Distance:",distance)
        history[-1]["distance"] = distance
        current_contract = revised_contract


    return {
        "status": "max_iterations_reached",
        "final_contract": current_contract,
        "optimization_suggestions":
        history[-1]["negotiation"].get("optimization_suggestions",{}),
        "history": history
    }

In [11]:
# user_request = """Build an indian tribal art painting classifier with around 95% accuracy under 6GB VRAM"""

# result = autonomous_contract_optimizer(
#     user_request,
#     model
# )

# print(result)
# final_contract = result["final_contract"]

# latest_negotiation = result["history"][-1]["negotiation"]

# optimization_suggestions = (
#     latest_negotiation["optimization_suggestions"]
# )

In [12]:
def planner_agent(contract,optimization_suggestions,model):
    
    PLANNER_SYSTEM_PROMPT = """
    You are an Autonomous ML Planning Agent.

    Your task is to generate an efficient and realistic machine learning training strategy.
    
    CRITICAL FRAMEWORK RULE:
    - For tasks lik image/vision or deep learning models, plan a PyTorch deep learning pipeline.
    - For tabular/regression tasks which can be solved by Traditional Machine learning, plan a Scikit-Learn or XGBoost pipeline (e.g., RandomForestRegressor, GradientBoostingClassifier).
    
    PYDANTIC COMPATIBILITY RULE:
    If you choose a traditional ML model (Scikit-Learn/XGBoost), you MUST still return valid types for deep learning fields to satisfy JSON schemas:
    - Set 'epochs' to 1
    - Set 'batch_size' to 0
    - Set 'learning_rate' to a standard float like 0.1
    - Set 'augmentations' to []
    - Set 'optimizer' and 'loss_function' to "N/A" or the closest equivalent (e.g., "mse").
    You must consider:
    - task type
    - dataset domain
    - target metrics
    - compute constraints
    - optimization suggestions

    Your goal is to maximize performance while respecting resource limitations.

    You may use:
    - lightweight architectures
    - mixed precision
    - transfer learning
    - gradient accumulation
    - reduced image resolution
    - efficient optimizers

    Return ONLY valid JSON.

    Required fields:
    - recommended_model
    - optimizer
    - learning_rate
    - batch_size
    - epochs
    - loss_function
    - augmentations
    - training_strategies
    - resource_optimizations

    Example:

    Input:
    {
      "contract": {
        "task_type": "image_classification",
        "dataset_domain": "medical imaging",
        "target_metric": "accuracy",
        "target_value": 0.90,
        "compute_constraint": "4GB VRAM"
      },

      "optimization_suggestions": {
        "recommended_model_family": "MobileNetV3",
        "mixed_precision": true,
        "gradient_accumulation": true
      }
    }

    Output:
    {
      "recommended_model": "MobileNetV3",
      "optimizer": "AdamW",
      "learning_rate": 0.0001,
      "batch_size": 8,
      "epochs": 15,
      "loss_function": "CrossEntropyLoss",
      "augmentations": [
        "horizontal_flip",
        "random_rotation",
        "normalize"
      ],
      "training_strategies": [
        "mixed_precision",
        "transfer_learning",
        "gradient_accumulation"
      ],
      "resource_optimizations": [
        "reduced_batch_size",
        "efficient_backbone"
      ]
    }
    """
    planner_input = {
        "contract": contract,
        "optimization_suggestions": optimization_suggestions
    }
    planner_text = json.dumps(planner_input,indent=2)
    messages = [
        {
            "role": "system",
            "content": PLANNER_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": planner_text
        }
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False,
        temperature=0.0,
        repetition_penalty=1.1
    )
    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
    assistant_response = response.split("assistant")[-1].strip()
    json_match = re.search(
        r"\{[\s\S]*\}",
        assistant_response
    )
    plan_json = json.loads(json_match.group())
    try:
        plan = TrainingPlan(**plan_json)
        return plan_json
    except Exception as e:
        print("Planner Pydantic Check Failed")
        print(e)
        return -1

In [13]:
from collections import Counter
# helper_fns
def indent_text(text, n=2):
    prefix = " " * n
    return "\n".join(prefix + line for line in text.split("\n"))

def extract_important_files(node,important_files=None):
    if important_files is None:
        important_files = []
    if node["kind"] == "directory":
        for child in node.get("children",[]):
            extract_important_files(child,important_files)
    else:
        ext = node.get("extension","")
        if ext in [".csv",".json",".txt",".yaml",".yml",".xml"]:

            important_files.append({
                "name":node.get("name"),
                "path":node.get("path"),
                "kind":node.get("kind")
            })
    return important_files

def find_image_statistics(node,stats=None):
    if stats is None:
        stats = {
            "image_count": 0,
            "extensions":Counter(),
            "sample_images": []
        }

    if node["kind"] == "directory":
        for child in node.get("children",[]):
            find_image_statistics(child,stats)
    else:
        if node["kind"] == "image":
            stats["image_count"] += 1
            ext = node.get("extension","")
            stats["extensions"][ext] += 1
            if len(stats["sample_images"]) < 5:
                stats["sample_images"].append(node.get("path"))
    return stats

def extract_csv_profiles(node,csv_profiles=None):
    if csv_profiles is None:
        csv_profiles = []
    if node["kind"] == "directory":
        for child in node.get("children",[]):
            extract_csv_profiles(child,csv_profiles)
    else:
        if node["kind"] == "csv":
            csv_profiles.append({
                "name":node.get("name"),
                "path":node.get("path"),
                "semantics_guess":node.get("semantics_guess"),
                "roles":node.get("roles"),
                "target_column_guess":node.get("target_column_guess"),
                "profile":node.get("profile")
            })
    return csv_profiles

# this will generate tree view
def build_tree_view(node,depth=0,max_depth=3):
    if depth > max_depth:
        return ""
    indent = "  " * depth
    output = ""
    if node["kind"] == "directory":
        output += (f"{indent} "f"{node['name']}\n")

        for child in node.get("children",[])[:15]:
            output += build_tree_view(child,depth + 1,max_depth)
    else:
        output += (f"{indent} "f"{node['name']}\n")
    return output



In [14]:
#dataset readme generator to send dataset structure and details which is an important issue 
# and send it in correct way works really well for the coding agent 



def generate_dataset_readme(metadata_json_path,output_readme_path=None):


    with open(metadata_json_path,"r",encoding="utf-8") as f:
        metadata = json.load(f)
    root = metadata["tree"]
    summary = metadata["summary"]

    important_files = extract_important_files(root)
    image_stats = find_image_statistics(root)
    csv_profiles = extract_csv_profiles(root)

    lines = []
    lines.append("# DATASET README")
    lines.append("")
    lines.append("## DATASET OVERVIEW")
    lines.append("")
    lines.append(f"- Root Path: "f"{metadata['root_path']}")
    lines.append(f"- Total Files: "f"{summary['total_files']}")
    lines.append(f"- Total Directories: "f"{summary['total_directories']}")
    lines.append("")
    # Adding file types
    lines.append("## FILE TYPE DISTRIBUTION")
    lines.append("")
    for ext, count in summary["extension_counts"].items():
        lines.append(f"- {ext}: {count}")

    lines.append("")

    
    # adding image summary if there is 
    

    if image_stats["image_count"] > 0:
        lines.append("## IMAGE DATA SUMMARY")
        lines.append("")
        lines.append(f"- Total Images: "f"{image_stats['image_count']}")
        lines.append("")
        lines.append("### Image Extensions")

        for ext, count in image_stats["extensions"].items():
            lines.append(f"- {ext}: {count}")

        lines.append("")
        lines.append("### Sample Image Paths")
        for path in image_stats["sample_images"]:
            lines.append(f"- {path}")
        lines.append("")

    #adding csv analysis
    if len(csv_profiles) > 0:
        lines.append("## CSV FILE ANALYSIS")
        lines.append("")
        for csv_info in csv_profiles:
            lines.append(f"### {csv_info['name']}")
            lines.append("")
            lines.append(f"- Path: "f"{csv_info['path']}")
            lines.append(f"- Semantic Guess: "f"{csv_info['semantics_guess']}")
            lines.append(f"- Target Column Guess: "f"{csv_info['target_column_guess']}" )
            profile = csv_info["profile"]
            shape = profile.get("shape",None)
            if shape:
                lines.append(f"- Shape: "f"{shape}")
            lines.append("")
            lines.append("#### Column Roles")
            roles = csv_info["roles"]
            for role_name, cols in roles.items():
                if len(cols) > 0:
                    lines.append(f"- {role_name}: "f"{cols}")
            lines.append("")
            sample_rows = profile.get("sample_rows",[])
            if len(sample_rows) > 0:
                lines.append("#### Sample Rows")
                lines.append("")
                for row in sample_rows[:3]:
                    lines.append(json.dumps(row,indent=2))
                    lines.append("")

    #important_files
    if len(important_files) > 0:
        lines.append("## IMPORTANT FILES")
        lines.append("")
        for file_info in important_files[:30]:
            lines.append(f"- {file_info['path']}")
        lines.append("")
    #dataset_tree
    lines.append("## DATASET STRUCTURE TREE")
    lines.append("")
    tree_view = build_tree_view(root)
    lines.append(tree_view)
    lines.append("")
    
    # coder_agent_instructions
    lines.append("## CODER AGENT INSTRUCTIONS")
    lines.append("")
    lines.append(
        """
Use this dataset information carefully.

Important Guidelines:
- Use correct dataset root paths.
- Use CSV annotation files if available.
- Use detected image columns correctly.
- Use target column guesses carefully.
- Join image paths relative to dataset root.
- Handle missing paths gracefully.
- Use train-validation split.
- Infer task type from semantic analysis.
- Use PyTorch best practices.
- Use mixed precision if GPU available.
"""
    )

    #joining all lines
    readme_text = "\n".join(lines)
    #saving readme_file
    if output_readme_path is not None:
        with open(output_readme_path,"w",encoding="utf-8") as f:
            f.write(readme_text)
        print(f"\nREADME saved to:\n"f"{output_readme_path}")
    return readme_text

# agent_context
def build_agent_context(metadata_json_path,max_chars=12000):
    readme = generate_dataset_readme(metadata_json_path)
    if len(readme) > max_chars:
        readme = readme[:max_chars]
    return f"""
DATASET README:

{readme}

IMPORTANT:
Use the dataset structure carefully.
Generate correct data loading code.
Handle file paths robustly.
Infer dataset semantics correctly.
"""


In [15]:

def generate_execution_spec(contract,dataset_readme,training_plan,runtime_dataset_info):

    dataset_understanding = {
        "dataset_readme":dataset_readme,
        "dataset_root":runtime_dataset_info.get("dataset_root"),
        "csv_path":runtime_dataset_info.get("csv_path"),
        "checkpoint_dir":runtime_dataset_info.get("checkpoint_dir")
    }
    #model_configuration
    model_spec = {
        "model_name":
        training_plan.get("recommended_model","resnet18"),
        "pretrained":True,
        "framework":"timm"
    }

    #training_configuration
    training_config = {
        "optimizer":training_plan.get("optimizer","AdamW"),
        "learning_rate":training_plan.get("learning_rate",1e-3),
        "batch_size":training_plan.get("batch_size",32 ),
        "epochs":training_plan.get("epochs",10),
        "loss_function":training_plan.get("loss_function","CrossEntropyLoss"),
        "scheduler":training_plan.get("scheduler","CosineAnnealingLR"),
        "weight_decay":training_plan.get("weight_decay",1e-4),
        "early_stopping":True
    }
    #evaluation_config
    evaluation_config = {
        "primary_metric":contract.get("target_metric","accuracy"),
        "target_value":contract.get("target_value",0.85),
        "validation_split":0.2,
        "save_best_model":True
    }

   #hardware_config
    hardware_config = {
        "compute_constraint":contract.get("compute_constraint","Unknown"),
        "mixed_precision":
        (
            "mixed_precision" in training_plan.get("training_strategies",[])
        ),
        "use_gpu":True,
        "multi_gpu":False
    }

    # optimization_config
    optimization_config = (training_plan.get("resource_optimizations",[]))
    #augumentations
    augmentation_config = (training_plan.get("augmentations",[]))
    #runtime_requir...

    runtime_requirements = {
        "environment":"kaggle",
        "working_directory":runtime_dataset_info.get("working_directory"),
        "kaggle_working_dir":runtime_dataset_info.get("kaggle_working_dir"),
        "checkpoint_dir":runtime_dataset_info.get("checkpoint_dir"),
        "requires_internet":False,
        "save_logs":True
    }

    agent_notes = [
        "Use dataset README carefully.",
        "Infer dataset structure from README.",
        "Use correct image paths.",
        "Use target column guesses from README.",
        "Handle missing files gracefully.",
        "Use train-validation split.",
        "Use mixed precision if GPU available.",
        "Save checkpoints periodically.",
        "Use Kaggle-compatible paths.",
        "Generate production-quality code."
    ]

    # final_exec_spec
    execution_spec = {
        "framework":"pytorch or sklearn",
        "task_type":contract.get("task_type","classification"),
        "dataset_understanding":dataset_understanding,
        "model_spec":model_spec,
        "training_config":training_config,
        "augmentation_config":augmentation_config,
        "optimization_config":optimization_config,
        "evaluation_config":evaluation_config,
        "hardware_config":hardware_config,
        "runtime_requirements":runtime_requirements,
        "agent_notes":agent_notes
    }
    try:
        validated_spec = ExecutionSpec(**execution_spec)
        return validated_spec.model_dump()
    except Exception as e:
        print("\nExecutionSpec validation failed:")
        print(e)
        return None

In [16]:
def build_runtime_dataset_info(dataset_path,dataset_schema):
    runtime_info = {
        "dataset_root":dataset_path,
        "working_directory":os.getcwd(),
        "kaggle_working_dir":"/kaggle/working",
        "checkpoint_dir":"/kaggle/working/autonomous_ml/checkpoints"
    }
    # CSV INFO
    csv_files = [
        f for f in os.listdir(dataset_path)
        if f.endswith(".csv")
    ]
    if len(csv_files) > 0:
        csv_path = os.path.join(dataset_path,csv_files[0])
        runtime_info["csv_path"] = csv_path
    return runtime_info



In [17]:
import os
import json
import math
import mimetypes
from datetime import datetime
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
try:
    from PIL import Image
except Exception:
    Image = None

SKIP_HIDDEN = True
MAX_SAMPLE_NAMES = 5
MAX_TEXT_PREVIEW_LINES = 10
MAX_TEXT_PREVIEW_CHARS = 4000
MAX_CSV_SAMPLE_ROWS = 5
MAX_FOLDER_RECURSION_DEPTH = 50



def to_jsonable(obj):
    """Recursively convert objects to JSON-serializable types."""
    if obj is None:
        return None
    if isinstance(obj, (str, int, bool)):
        return obj
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
        return obj
    if isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    if isinstance(obj, (np.floating, np.float64, np.float32)):
        val = float(obj)
        if math.isnan(val) or math.isinf(val):
            return None
        return val
    if isinstance(obj, (pd.Timestamp, datetime)):
        return obj.isoformat()
    if isinstance(obj, pd.Timedelta):
        return str(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (list, tuple, set)):
        return [to_jsonable(x) for x in obj]
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            out[str(k)] = to_jsonable(v)
        return out
    if pd.isna(obj):
        return None
    return str(obj)

def save_json(data, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True) if os.path.dirname(output_path) else None
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(to_jsonable(data), f, indent=2, ensure_ascii=False)

#another set of helper fns...
def is_hidden(name):
    return name.startswith(".")

def safe_listdir(path):
    try:
        return os.listdir(path)
    except Exception:
        return []

def file_size_bytes(path):
    try:
        return os.path.getsize(path)
    except Exception:
        return None

def format_mime(path):
    mime, _ = mimetypes.guess_type(path)
    return mime

def extension_of(path):
    return os.path.splitext(path)[1].lower()

def iso_mtime(path):
    try:
        return datetime.fromtimestamp(os.path.getmtime(path)).isoformat()
    except Exception:
        return None

def sample_names(names, k=MAX_SAMPLE_NAMES):
    return names[:k]

def read_text_preview(path, max_lines=MAX_TEXT_PREVIEW_LINES, max_chars=MAX_TEXT_PREVIEW_CHARS):
    try:
        lines = []
        total_lines = 0
        total_chars = 0
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            for line in f:
                total_lines += 1
                if len(lines) < max_lines and total_chars < max_chars:
                    clean = line.rstrip("\n")
                    lines.append(clean)
                    total_chars += len(clean) + 1
        return {
            "line_count": total_lines,
            "preview_lines": lines,
        }
    except Exception as e:
        return {
            "line_count": None,
            "preview_lines": [],
            "error": str(e),
        }

#image_profiling...
def profile_image_file(path):
    result = {
        "kind": "image",
        "path": path,
        "name": os.path.basename(path),
        "extension": extension_of(path),
        "size_bytes": file_size_bytes(path),
        "modified_time": iso_mtime(path),
        "mime_type": format_mime(path),
    }
    if Image is not None:
        try:
            with Image.open(path) as img:
                result["image_info"] = {
                    "width": img.width,
                    "height": img.height,
                    "mode": img.mode,
                    "format": img.format,
                }
        except Exception as e:
            result["image_info"] = {"error": str(e)}
    else:
        result["image_info"] = {"error": "PIL not available"}
    return result

#text_profiling
def profile_text_file(path):
    preview = read_text_preview(path)
    return {
        "kind": "text",
        "path": path,
        "name": os.path.basename(path),
        "extension": extension_of(path),
        "size_bytes": file_size_bytes(path),
        "modified_time": iso_mtime(path),
        "mime_type": format_mime(path),
        "line_count": preview.get("line_count"),
        "preview_lines": preview.get("preview_lines", []),
        "error": preview.get("error"),
    }

#csv_profiling
def guess_csv_semantics(df):
    cols = [str(c).lower() for c in df.columns]
    image_keywords = ["image", "img", "path", "filepath", "filename", "file"]
    text_keywords = ["text", "sentence", "review", "caption", "content", "story"]
    label_keywords = ["label", "class", "target", "y", "output", "category"]
    has_image_col = any(any(k in c for k in image_keywords) for c in cols)
    has_text_col = any(any(k in c for k in text_keywords) for c in cols)
    has_label_col = any(any(k in c for k in label_keywords) for c in cols)
    if has_image_col and has_label_col:
        return "image_annotations_csv"
    if has_text_col and has_label_col:
        return "text_classification_csv"
    if has_label_col:
        return "tabular_or_classification_csv"
    return "tabular_csv"

def column_roles(df):
    roles = {
        "image_columns": [],
        "text_columns": [],
        "label_columns": [],
        "numeric_columns": [],
        "categorical_columns": [],
        "other_columns": [],
    }
    image_keywords = ["image", "img", "path", "filepath", "filename", "file"]
    text_keywords = ["text", "sentence", "review", "caption", "content", "story"]
    label_keywords = ["label", "class", "target", "y", "output", "category"]
    for col in df.columns:
        name = str(col).lower()
        dtype = str(df[col].dtype)

        if any(k in name for k in image_keywords):
            roles["image_columns"].append(col)
        elif any(k in name for k in text_keywords):
            roles["text_columns"].append(col)
        elif any(k in name for k in label_keywords):
            roles["label_columns"].append(col)
        elif pd.api.types.is_numeric_dtype(df[col]):
            roles["numeric_columns"].append(col)
        elif dtype in ("object", "category", "string"):
            roles["categorical_columns"].append(col)
        else:
            roles["other_columns"].append(col)
    return roles

def safe_describe_df(df):
    try:
        desc = df.describe(include="all").replace({np.nan: None})
        return desc.to_dict()
    except Exception as e:
        return {"error": str(e)}

def safe_numeric_describe(df):
    try:
        num = df.describe(include=[np.number]).replace({np.nan: None})
        return num.to_dict()
    except Exception as e:
        return {"error": str(e)}

def dataframe_profile(df):
    missing = df.isnull().sum()
    missing_pct = (missing / max(len(df), 1) * 100).round(4)

    stats = {
        "shape": [int(df.shape[0]), int(df.shape[1])],
        "columns": [str(c) for c in df.columns],
        "dtypes": {str(c): str(df[c].dtype) for c in df.columns},
        "missing_values": {str(c): int(missing[c]) for c in df.columns},
        "missing_percent": {str(c): float(missing_pct[c]) for c in df.columns},
        "sample_rows": df.head(MAX_CSV_SAMPLE_ROWS).replace({np.nan: None}).to_dict(orient="records"),
        "describe_all": safe_describe_df(df),
        "describe_numeric": safe_numeric_describe(df),
        "memory_usage_bytes": int(df.memory_usage(deep=True).sum()),
    }
    return stats

def profile_csv_file(path):
    try:
        df = pd.read_csv(path, low_memory=False)
    except Exception as e:
        return {
            "kind": "csv",
            "path": path,
            "name": os.path.basename(path),
            "extension": extension_of(path),
            "size_bytes": file_size_bytes(path),
            "modified_time": iso_mtime(path),
            "mime_type": format_mime(path),
            "error": f"Failed to read CSV: {e}",
        }

    roles = column_roles(df)
    semantics = guess_csv_semantics(df)

    target_candidates = roles["label_columns"][:1]
    target_column = target_candidates[0] if target_candidates else None

    profile = {
        "kind": "csv",
        "path": path,
        "name": os.path.basename(path),
        "extension": extension_of(path),
        "size_bytes": file_size_bytes(path),
        "modified_time": iso_mtime(path),
        "mime_type": format_mime(path),
        "semantics_guess": semantics,
        "profile": dataframe_profile(df),
        "roles": roles,
        "target_column_guess": target_column,
        "unique_target_values_guess": int(df[target_column].nunique()) if target_column and target_column in df.columns else None,
    }
    return profile

#generic_files_profiling...

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp", ".gif"}
TEXT_EXTENSIONS = {".txt", ".md", ".rtf", ".csv", ".tsv", ".json", ".xml", ".yaml", ".yml"}

def profile_file(path):
    ext = extension_of(path)
    if ext in IMAGE_EXTENSIONS:
        return profile_image_file(path)
    if ext == ".csv":
        return profile_csv_file(path)
    if ext in {".txt", ".md", ".log", ".json", ".xml", ".yaml", ".yml", ".tsv"}:
        return profile_text_file(path)
    return {
        "kind": "file",
        "path": path,
        "name": os.path.basename(path),
        "extension": ext,
        "size_bytes": file_size_bytes(path),
        "modified_time": iso_mtime(path),
        "mime_type": format_mime(path),
    }
    
#directory_profiling
def scan_node(path, depth=0):
    if depth > MAX_FOLDER_RECURSION_DEPTH:
        return {
            "kind": "directory",
            "path": path,
            "name": os.path.basename(path),
            "error": "Max recursion depth reached",
        }
    if os.path.isfile(path):
        return profile_file(path)
    node = {
        "kind": "directory",
        "path": path,
        "name": os.path.basename(path) if os.path.basename(path) else path,
        "children": [],
        "summary": {},
    }
    entries = safe_listdir(path)
    if SKIP_HIDDEN:
        entries = [e for e in entries if not is_hidden(e)]
    child_paths = [os.path.join(path, e) for e in entries]
    files = []
    dirs = []
    for p in child_paths:
        if os.path.isdir(p):
            dirs.append(p)
        elif os.path.isfile(p):
            files.append(p)
        
    #folder_level_summary
    ext_counter = Counter()
    file_names_by_ext = defaultdict(list)
    kind_counter = Counter()
    for f in files:
        ext = extension_of(f)
        ext_counter[ext] += 1
        file_names_by_ext[ext].append(os.path.basename(f))
        kind_counter["image"] += int(ext in IMAGE_EXTENSIONS)
        kind_counter["csv"] += int(ext == ".csv")
        kind_counter["text"] += int(ext in {".txt", ".md", ".log", ".json", ".xml", ".yaml", ".yml", ".tsv"})
    
    #summary_addition
    node["summary"] = {
        "file_count": len(files),
        "folder_count": len(dirs),
        "extension_counts": dict(ext_counter),
        "sample_file_names_by_extension": {
            ext: sample_names(names, MAX_SAMPLE_NAMES)
            for ext, names in file_names_by_ext.items()
        },
        "kind_counts_guess": dict(kind_counter),
    }

    #adding_file_nodes
    for f in files:
        node["children"].append(profile_file(f))
    # Recurse into subdirectories
    for d in dirs:
        node["children"].append(scan_node(d, depth=depth + 1))
    return node

#root_metadata
def build_dataset_metadata(input_path, output_json_path=None):
    input_path = os.path.abspath(input_path)
    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Path not found: {input_path}")
    root_node = scan_node(input_path)
    # Aggregate a top-level summary
    def aggregate(node):
        total_files = 0
        total_dirs = 0
        ext_counter = Counter()
        kind_counter = Counter()

        def walk(n):
            nonlocal total_files, total_dirs, ext_counter, kind_counter
            if isinstance(n, dict):
                if n.get("kind") == "directory":
                    total_dirs += 1
                    summary = n.get("summary", {})
                    for ext, cnt in (summary.get("extension_counts") or {}).items():
                        ext_counter[ext] += int(cnt)
                    for k, v in (summary.get("kind_counts_guess") or {}).items():
                        kind_counter[k] += int(v)

                    for ch in n.get("children", []):
                        walk(ch)
                else:
                    total_files += 1
                    ext = n.get("extension")
                    if ext is not None:
                        ext_counter[ext] += 1
                    kind = n.get("kind")
                    if kind:
                        kind_counter[kind] += 1

        walk(node)
        return {
            "total_files": total_files,
            "total_directories": total_dirs,
            "extension_counts": dict(ext_counter),
            "kind_counts_guess": dict(kind_counter),
        }

    meta = {
        "generated_at": datetime.now().isoformat(),
        "root_path": input_path,
        "summary": aggregate(root_node),
        "tree": root_node,
    }

    if output_json_path is not None:
        save_json(meta, output_json_path)
    return meta

#agent_summary_extraction

def build_agent_summary(metadata):
    """
    A compact summary you can feed to your local LLM or Gemini later.
    This keeps the prompt small and avoids quota waste.
    """
    root = metadata.get("tree", {})
    summary = metadata.get("summary", {})
    return {
        "root_path": metadata.get("root_path"),
        "generated_at": metadata.get("generated_at"),
        "total_files": summary.get("total_files"),
        "total_directories": summary.get("total_directories"),
        "extension_counts": summary.get("extension_counts"),
        "root_summary": root.get("summary", {}),
        "children_preview": [
            {
                "kind": ch.get("kind"),
                "name": ch.get("name"),
                "path": ch.get("path"),
                "summary": ch.get("summary") if ch.get("kind") == "directory" else None,
                "extension": ch.get("extension") if ch.get("kind") != "directory" else None,
            }
            for ch in root.get("children", [])[:10]
        ],
    }


In [18]:
# # Change this to a folder, CSV file, text file, or image file.
# INPUT_PATH = "/kaggle/input/datasets/ajg117/indian-paintings-dataset"

# # Optional output path
# OUTPUT_JSON = "/kaggle/working/dataset_metadata.json"

# metadata = build_dataset_metadata(INPUT_PATH, OUTPUT_JSON)

# print("Saved metadata to:", OUTPUT_JSON)
# print(json.dumps(build_agent_summary(metadata), indent=2))

In [19]:
#kaggle compatible self_improving_coder

import os
import re
import json
import time
import shutil
import subprocess
from copy import deepcopy
from datetime import datetime
import google.generativeai as genai


GEMINI_API_KEY = "geminin_api_key"
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-3.1-flash-lite")
#rate_limits_

LAST_API_CALL = 0
MIN_REQUEST_INTERVAL = 60

def gemini_call(prompt):
    global LAST_API_CALL
    elapsed = (time.time() - LAST_API_CALL)
    if elapsed < MIN_REQUEST_INTERVAL:
        wait_time = ( MIN_REQUEST_INTERVAL - elapsed)
        print(f"\nWaiting {wait_time:.1f}s " "for Gemini rate limit...")
        time.sleep(wait_time)

    response = gemini_model.generate_content(prompt)
    LAST_API_CALL = time.time()
    return response



# setting_workspace
BASE_DIR = ("/kaggle/working/autonomous_ml")
EXPERIMENTS_DIR = os.path.join(BASE_DIR,"experiments")
BEST_MODEL_DIR = os.path.join(BASE_DIR,"best_model")
REPORTS_DIR = os.path.join(BASE_DIR,"reports")
CHECKPOINTS_DIR = os.path.join(BASE_DIR,"checkpoints")
os.makedirs(EXPERIMENTS_DIR,exist_ok=True)
os.makedirs(BEST_MODEL_DIR,exist_ok=True)
os.makedirs(REPORTS_DIR,exist_ok=True)
os.makedirs(CHECKPOINTS_DIR,exist_ok=True)

#helper_fns_again
def timestamp():
    return datetime.now().strftime("%Y%m%d_%H%M%S")
import re

def clean_code(text):
    # Extract code block if it exists
    match = re.search(r'```python\s*(.*?)\s*```', text, re.DOTALL)
    if match:
        return match.group(1)
    # Fallback if the LLM forgets the markdown tags
    return text.strip()
#save_json
def save_json(data, path):
    with open(path, "w") as f:
        json.dump(data,f,indent=2)



#instllation of missing package as that was also an issue while working
def install_missing_packages(stderr):
    patterns = [
        r"No module named '([^']+)'",
        r'No module named "([^"]+)"'
    ]
    missing = []
    for pattern in patterns:
        matches = re.findall(pattern,stderr)
        missing.extend(matches)
    if len(missing) == 0:
        return False
    mapping = {
        "PIL": "Pillow",
        "cv2": "opencv-python",
        "sklearn": "scikit-learn"
    }
    packages = []
    for module in missing:
        pkg = mapping.get(module,module)
        packages.append(pkg)
    packages = list(set(packages))
    print("\nInstalling:",packages)
    cmd = [
        "pip",
        "install",
        "-q"
    ] + packages
    subprocess.run(cmd)
    return True

#coding_agent
def coding_agent(execution_spec,training_plan,dataset_readme,runtime_dataset_info):
    prompt = f"""
You are an expert ML engineer.

Generate COMPLETE runnable Kaggle-compatible training code.

CRITICAL FRAMEWORK RULE:
Analyze the TRAINING PLAN and EXECUTION SPEC below.
- If the 'recommended_model' is a Deep Learning architecture (e.g., ResNet, EfficientNet, PyTorch models), write a PyTorch pipeline.
- If the 'recommended_model' is a traditional ML algorithm (e.g., RandomForest, XGBoost, LogisticRegression), write a Scikit-Learn/XGBoost pipeline.


IMPORTANT:
- Return ONLY Python code
- No markdown
- No explanations
- Use mixed precision if enabled (PyTorch only)
- Save the best model and checkpoints (use torch.save for PyTorch, or joblib/pickle for Scikit-Learn)
- Use train-validation split
- Use modern best practices
- Use correct dataset paths
- Use dataset README carefully

EXECUTION SPEC:
{json.dumps(execution_spec, indent=2)}

TRAINING PLAN:
{json.dumps(training_plan, indent=2)}

RUNTIME INFO:
{json.dumps(runtime_dataset_info, indent=2)}

DATASET README:
{dataset_readme}
"""

    response = gemini_call(prompt)
    code = clean_code(response.text)
    return code

#debugging_agent
def debugging_agent(code, stderr, execution_spec, training_plan, dataset_readme):
    prompt = f"""
You are an expert ML debugging engineer.

The following machine learning training pipeline failed with an error. 

Step 1: Briefly analyze the TRACEBACK to identify the root cause (e.g., API version mismatch, incorrect data shapes, missing arguments).
Step 2: Formulate a generalized fix that relies on stable, core API arguments.
Step 3: Output the COMPLETE, fully corrected Python script. You must wrap the code in a standard markdown python block (```python ... 
```).

FAILED CODE:
{code}

TRACEBACK:
{stderr}

EXECUTION SPEC:
{json.dumps(execution_spec, indent=2)}

TRAINING PLAN:
{json.dumps(training_plan, indent=2)}

DATASET README:
{dataset_readme}
"""
    response = gemini_call(prompt)
    fixed_code = clean_code(response.text)
    return fixed_code
#training_log_parser
# def parse_training_log(log_text):
#     metrics = {
#         "best_val_accuracy": 0.0,
#         "final_val_accuracy": 0.0,
#         "epochs_completed": 0,
#         "accuracy_history": [],
#         "loss_history": [],
#         "training_completed": False
#     }
#     #accuracy_extraction
#     acc_patterns = [
#         r"Val Accuracy[:=]\s*([0-9.]+)",
#         r"Validation Accuracy[:=]\s*([0-9.]+)",
#         r"val_acc[:=]\s*([0-9.]+)",
#         r"accuracy[:=]\s*([0-9.]+)"
#     ]
#     accuracies = []
#     for pattern in acc_patterns:
#         matches = re.findall(pattern,log_text,re.IGNORECASE)
#         for m in matches:
#             try:
#                 val = float(m)
#                 if val <= 1.0:
#                     val *= 100
#                 accuracies.append(val)
#             except:
#                 pass
#     if len(accuracies) > 0:
#         metrics["accuracy_history"] = accuracies
#         metrics["best_val_accuracy"] = max(accuracies)
#         metrics["final_val_accuracy"] = accuracies[-1]

#     epoch_matches = re.findall(r"Epoch\s+(\d+)",log_text)
#     if len(epoch_matches) > 0:
#         metrics["epochs_completed"] = int(epoch_matches[-1])

#     if ("training complete" in log_text.lower() or"finished training" in log_text.lower()):
#         metrics["training_completed"] = True
#     return metrics

# #performance_anaalysis
# def performance_analyzer_agent(metrics,execution_spec,training_plan,contract):

#     prompt = f"""
# You are an ML performance analysis expert.

# Analyze training performance.

# Return ONLY valid JSON.

# Required fields:
# - issues_detected
# - recommendations
# - should_continue_optimization
# - estimated_improvement_potential

# METRICS:
# {json.dumps(metrics, indent=2)}

# EXECUTION SPEC:
# {json.dumps(execution_spec, indent=2)}

# TRAINING PLAN:
# {json.dumps(training_plan, indent=2)}

# CONTRACT:
# {json.dumps(contract, indent=2)}
# """
#     response = gemini_call(prompt)
#     text = clean_code(response.text)
#     json_match = re.search(r"\{[\s\S]*\}",text)
#     analysis = json.loads(json_match.group())
#     return analysis
    
# #training_improvemet_agent
# def training_improvement_agent(training_plan,performance_analysis,metrics):
#     prompt = f"""
# You are an ML optimization expert.

# Improve the training plan.

# Goals:
# - increase validation accuracy
# - preserve stability
# - use better optimizers
# - improve augmentations
# - improve regularization

# Return ONLY valid JSON.

# Required:
# - updated_training_plan
# - changes_made

# CURRENT TRAINING PLAN:
# {json.dumps(training_plan, indent=2)}

# PERFORMANCE ANALYSIS:
# {json.dumps(performance_analysis, indent=2)}

# METRICS:
# {json.dumps(metrics, indent=2)}
# """
#     response = gemini_call(prompt)
#     text = clean_code(response.text)
#     json_match = re.search(r"\{[\s\S]*\}",text)
#     improvement = json.loads(json_match.group())
#     return improvement

# #executing code
# def execute_code(code_path,experiment_dir):
#     result = subprocess.run(
#         ["python", code_path],
#         capture_output=True,
#         text=True
#     )
#     stdout = result.stdout
#     stderr = result.stderr
#     log_path = os.path.join(experiment_dir,"execution_log.txt")

#     with open(log_path, "w") as f:
#         f.write(stdout)
#         f.write("\n\n===== STDERR =====\n\n")
#         f.write(stderr)
#     return {
#         "return_code":result.returncode,
#         "stdout":stdout,
#         "stderr":stderr,
#         "log_path":log_path
#     }

# GLOBAL_BEST_ACCURACY = 0.0
# def update_best_experiment(metrics,experiment_dir,training_plan,execution_spec):
#     global GLOBAL_BEST_ACCURACY
#     current_acc = metrics["best_val_accuracy"]
#     if current_acc > GLOBAL_BEST_ACCURACY:
#         GLOBAL_BEST_ACCURACY = current_acc
#         print(f"\nNEW BEST MODEL: "f"{current_acc:.2f}%")
#         best_exp_dir = os.path.join(BEST_MODEL_DIR,"best_experiment")
#         if os.path.exists(best_exp_dir):
#             shutil.rmtree(best_exp_dir)
#         shutil.copytree(experiment_dir,best_exp_dir)
#         #saving_metadata
#         save_json(
#             {
#                 "best_accuracy":current_acc,
#                 "training_plan":training_plan,
#                 "execution_spec":execution_spec
#             },
#             os.path.join(BEST_MODEL_DIR,"best_metadata.json")
#         )





/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)


In [27]:
import re
import json
import os
import shutil
import subprocess

# --- NEW: Dynamic Metric Evaluator ---
def is_metric_better(current_metric, best_metric, metric_name):
    """Dynamically determines if a metric improved based on its type."""
    if best_metric is None:
        return True
        
    metric_name = str(metric_name).lower().strip()
    lower_is_better_keywords = ['rmse', 'mse', 'mae', 'loss', 'error', 'bce', 'ce']
    
    if any(keyword in metric_name for keyword in lower_is_better_keywords):
        return current_metric < best_metric
    return current_metric > best_metric


def parse_training_log(log_text):
    """Parses standard output text for generic metrics if metrics.json fails."""
    metrics = {
        "best_primary_metric": None,
        "final_primary_metric": None,
        "primary_metric_name": "unknown",
        "epochs_completed": 0,
        "primary_metric_history": [],
        "training_completed": False
    }
    
    # Check for Error Metrics (Regression / Loss)
    error_patterns = [
        r"Val(?:idation)?\s*RMSE[:=]\s*([0-9.]+)",
        r"Val(?:idation)?\s*MSE[:=]\s*([0-9.]+)",
        r"Val(?:idation)?\s*Loss[:=]\s*([0-9.]+)",
        r"rmse[:=]\s*([0-9.]+)"
    ]
    
    # Check for Success Metrics (Classification)
    acc_patterns = [
        r"Val(?:idation)?\s*Accuracy[:=]\s*([0-9.]+)",
        r"val_acc[:=]\s*([0-9.]+)",
        r"accuracy[:=]\s*([0-9.]+)"
    ]
    
    found_metrics = []
    metric_type = "unknown"

    # 1. Search for error metrics first
    for pattern in error_patterns:
        matches = re.findall(pattern, log_text, re.IGNORECASE)
        if matches:
            found_metrics.extend([float(m) for m in matches])
            # Assign a generic name based on the pattern match
            metric_type = "rmse" if "rmse" in pattern.lower() else "loss"
            break

    # 2. If no error metrics, search for accuracy metrics
    if not found_metrics:
        for pattern in acc_patterns:
            matches = re.findall(pattern, log_text, re.IGNORECASE)
            if matches:
                for m in matches:
                    try:
                        val = float(m)
                        # Keep percentage formatting for accuracy
                        if val <= 1.0:
                            val *= 100
                        found_metrics.append(val)
                    except ValueError:
                        pass
                metric_type = "accuracy"
                break

    # 3. Populate metrics dictionary
    if len(found_metrics) > 0:
        metrics["primary_metric_history"] = found_metrics
        metrics["final_primary_metric"] = found_metrics[-1]
        metrics["primary_metric_name"] = metric_type
        
        # Min or Max depending on type
        if "accuracy" in metric_type:
            metrics["best_primary_metric"] = max(found_metrics)
        else:
            metrics["best_primary_metric"] = min(found_metrics)

    # Extract Epochs
    epoch_matches = re.findall(r"Epoch\s+(\d+)", log_text, re.IGNORECASE)
    if len(epoch_matches) > 0:
        metrics["epochs_completed"] = int(epoch_matches[-1])

    # Extract Completion Status
    if "training complete" in log_text.lower() or "finished training" in log_text.lower() or "saved" in log_text.lower():
        metrics["training_completed"] = True
        
    return metrics


def performance_analyzer_agent(metrics, execution_spec, training_plan, contract):
    prompt = f"""
You are an ML performance analysis expert.

Analyze training performance based on the provided metrics.
Note: If the primary metric is an error metric (e.g., RMSE, Loss), lower is better. If it is a success metric (e.g., Accuracy, F1), higher is better.

Return ONLY valid JSON.

Required fields:
- issues_detected
- recommendations
- should_continue_optimization
- estimated_improvement_potential

METRICS:
{json.dumps(metrics, indent=2)}

EXECUTION SPEC:
{json.dumps(execution_spec, indent=2)}

TRAINING PLAN:
{json.dumps(training_plan, indent=2)}

CONTRACT:
{json.dumps(contract, indent=2)}
"""
    response = gemini_call(prompt)
    text = clean_code(response.text)
    json_match = re.search(r"\{[\s\S]*\}", text)
    analysis = json.loads(json_match.group())
    return analysis
    

def training_improvement_agent(training_plan, performance_analysis, metrics):
    prompt = f"""
You are an ML optimization expert.

Improve the training plan based on the performance analysis.

Goals:
- Optimize the target evaluation metric (minimize error or maximize accuracy)
- Preserve pipeline stability
- Use better optimizers or hyperparameter tuning
- Improve augmentations/feature engineering
- Improve regularization

Return ONLY valid JSON.

Required:
- updated_training_plan
- changes_made

CURRENT TRAINING PLAN:
{json.dumps(training_plan, indent=2)}

PERFORMANCE ANALYSIS:
{json.dumps(performance_analysis, indent=2)}

METRICS:
{json.dumps(metrics, indent=2)}
"""
    response = gemini_call(prompt)
    text = clean_code(response.text)
    json_match = re.search(r"\{[\s\S]*\}", text)
    improvement = json.loads(json_match.group())
    return improvement


def execute_code(code_path, experiment_dir):
    result = subprocess.run(
        ["python", code_path],
        capture_output=True,
        text=True
    )
    stdout = result.stdout
    stderr = result.stderr
    log_path = os.path.join(experiment_dir, "execution_log.txt")

    with open(log_path, "w") as f:
        f.write(stdout)
        f.write("\n\n===== STDERR =====\n\n")
        f.write(stderr)
        
    return {
        "return_code": result.returncode,
        "stdout": stdout,
        "stderr": stderr,
        "log_path": log_path
    }


GLOBAL_BEST_METRIC = None

def update_best_experiment(metrics, experiment_dir, training_plan, execution_spec):
    global GLOBAL_BEST_METRIC
    
    current_metric = metrics.get("best_primary_metric")
    metric_name = metrics.get("primary_metric_name", "unknown")
    
    # Failsafe if parsing failed
    if current_metric is None:
        print("\nNo valid primary metric found. Skipping update.")
        return

    # Check if the metric improved using the dynamic function
    if is_metric_better(current_metric, GLOBAL_BEST_METRIC, metric_name):
        GLOBAL_BEST_METRIC = current_metric
        print(f"\n NEW BEST MODEL! {metric_name.upper()}: {current_metric:.4f}")
        
        best_exp_dir = os.path.join(BEST_MODEL_DIR, "best_experiment")
        if os.path.exists(best_exp_dir):
            shutil.rmtree(best_exp_dir)
        shutil.copytree(experiment_dir, best_exp_dir)
        
        # Save updated metadata
        save_json(
            {
                "best_metric": current_metric,
                "metric_name": metric_name,
                "training_plan": training_plan,
                "execution_spec": execution_spec
            },
            os.path.join(BEST_MODEL_DIR, "best_metadata.json")
        )
    else:
        # Optional logging so you know it was checked
        best_display = GLOBAL_BEST_METRIC if GLOBAL_BEST_METRIC is not None else "None"
        print(f"\nModel did not improve. Current {metric_name.upper()}: {current_metric:.4f} | Best: {best_display}")

In [28]:
from copy import deepcopy
import os
import json

def autonomous_research_loop(execution_spec, training_plan, dataset_readme, runtime_dataset_info,
                             contract, max_research_iterations=5, max_debug_attempts=10):
    current_plan = deepcopy(training_plan)
    best_metrics = None
    
    for research_iter in range(max_research_iterations):
        print(f"\n\n{'='*60}")
        print(f"RESEARCH ITERATION {research_iter+1}")
        print(f"{'='*60}\n")
        
        # Creating experiment dir
        experiment_dir = os.path.join(EXPERIMENTS_DIR, f"exp_{research_iter+1:03d}")
        os.makedirs(experiment_dir, exist_ok=True)

        save_json(current_plan, os.path.join(experiment_dir, "training_plan.json"))
        
        # Code generation
        code = coding_agent(execution_spec, current_plan, dataset_readme, runtime_dataset_info)
        code_path = os.path.join(experiment_dir, "generated_pipeline.py")
        with open(code_path, "w") as f:
            f.write(code)
            
        # Debug loop
        execution_result = None
        for debug_attempt in range(max_debug_attempts):
            print(f"\nExecution Attempt {debug_attempt+1}")
            execution_result = execute_code(code_path, experiment_dir)
            
            if execution_result["return_code"] == 0:
                print("\nExecution Success!")
                break
                
            # Installing missing package
            installed = install_missing_packages(execution_result["stderr"])
            if installed:
                continue
            
            print("\nDebugging code...")
            code = debugging_agent(
                code,
                execution_result["stderr"],
                execution_spec,
                current_plan,
                dataset_readme
            )
            with open(code_path, "w") as f:
                f.write(code)
                
        # Parse logs
        full_log = (execution_result["stdout"] + "\n" + execution_result["stderr"])
        
        # Ensure metrics are read correctly either from JSON (if saved by agent) or parsed logs
        metrics_file = os.path.join(experiment_dir, "metrics.json")
        if os.path.exists(metrics_file):
            try:
                with open(metrics_file, "r") as f:
                    metrics = json.load(f)
            except Exception:
                metrics = parse_training_log(full_log)
        else:
            metrics = parse_training_log(full_log)
            
        print("\nMETRICS:")
        print(json.dumps(metrics, indent=2))
        
        # Saving metrics
        save_json(metrics, metrics_file)

        update_best_experiment(metrics, experiment_dir, current_plan, execution_spec)

        # --- NEW: Dynamic Contract Check ---
        current_best = metrics.get("best_primary_metric")
        metric_name = metrics.get("primary_metric_name", "unknown").lower()
        target_value = contract.get("target_value", 0)
        
        target_achieved = False
        display_target = target_value
        
        if current_best is not None:
            lower_is_better_keywords = ['rmse', 'mse', 'mae', 'loss', 'error', 'bce', 'ce']
            
            # Scenario A: Minimizing an error metric
            if any(keyword in metric_name for keyword in lower_is_better_keywords):
                target_achieved = current_best <= target_value
                
            # Scenario B: Maximizing a success metric
            else:
                # Handle percentage scaling dynamically 
                display_target = target_value * 100 if (target_value <= 1.0 and current_best > 1.0) else target_value
                target_achieved = current_best >= display_target

        if target_achieved:
            print(f"\n TARGET ACHIEVED!")
            print(f"Reached {metric_name.upper()}: {current_best:.4f} (Target: {display_target})")
            return {
                "status": "target_achieved",
                "metrics": metrics,
                "best_metric": GLOBAL_BEST_METRIC
            }
            
        # Performance analysis
        analysis = performance_analyzer_agent(metrics, execution_spec, current_plan, contract)
        print("\nPERFORMANCE ANALYSIS:")
        print(json.dumps(analysis, indent=2))
        
        # Stopping condition
        if not analysis.get("should_continue_optimization", True):
            print("\nOptimization stopped.")
            break
            
        # Improving training plan
        improvement = training_improvement_agent(current_plan, analysis, metrics)
        print("\nTRAINING IMPROVEMENTS:")
        print(json.dumps(improvement, indent=2))
        
        current_plan = improvement.get("updated_training_plan", current_plan)
        best_metrics = metrics
        
    return {
        "status": "finished",
        "best_metric": GLOBAL_BEST_METRIC,
        "best_metrics": best_metrics
    }

In [29]:

def autonomus_pipeline(user_request,dataset_path,metadata_json_path,readme_path):
    result = autonomous_contract_optimizer(user_request,model)
    final_contract = result["final_contract"]
    latest_negotiation = result["history"][-1]["negotiation"]
    optimization_suggestions = (latest_negotiation["optimization_suggestions"])
    plan = planner_agent(final_contract,optimization_suggestions,model)
    print(plan)
    print("\n\n\n==================================\n")
    metadata = build_dataset_metadata(dataset_path, metadata_json_path)
    # print(json.dumps(build_agent_summary(metadata), indent=2))
    readme = generate_dataset_readme(
        metadata_json_path=metadata_json_path,
        output_readme_path=readme_path
    )
    coder_context = build_agent_context(metadata_json_path)
    runtime_dataset_info = build_runtime_dataset_info(
        dataset_path=dataset_path,
        dataset_schema=None
    )
    execution_spec = generate_execution_spec(result["final_contract"],coder_context,plan,
        runtime_dataset_info
    )
    print(execution_spec)
    print("\n\n\n==================================\n")
    result = autonomous_research_loop(
        execution_spec=execution_spec,
        training_plan=plan,
        dataset_readme=coder_context,
        runtime_dataset_info=runtime_dataset_info,
        contract=final_contract,
        max_research_iterations=5,
        max_debug_attempts=10
    )
    print(result)

    


In [30]:
user_request = """Build a house price prediction model with rmse <=1 using 8GB VRAM"""
dataset_path="/kaggle/input/datasets/anmolkumar/house-price-prediction-challenge"
readme_path="/kaggle/working/dataset_readme.txt"
metadata_json_path="/kaggle/working/dataset_metadata.json"
#result = autonomous_contract_optimizer(user_request,model)
autonomus_pipeline(user_request,dataset_path,metadata_json_path,readme_path)



=== Iteration 1 ===
Feasibility: {'feasible': False, 'reason': 'The required RMSE of 1.0 is too low given the limited 8GB VRAM, which may not support complex models needed for accurate predictions.', 'suggested_target': 2.5}

=== Iteration 2 ===
Feasibility: {'feasible': True, 'reason': 'Given the dataset domain and regression task, achieving an RMSE of 2.5 is plausible with 8GB VRAM.', 'suggested_target': 2.5}

Feasible contract found!
{'recommended_model': 'XGBoost', 'optimizer': 'N/A', 'learning_rate': 0.1, 'batch_size': 0, 'epochs': 1, 'loss_function': 'mse', 'augmentations': [], 'training_strategies': ['gradient_booster_gblinear', 'feature_selection', 'early_stopping'], 'resource_optimizations': ['optimized_tree_structure', 'reduced_memory_usage']}





README saved to:
/kaggle/working/dataset_readme.txt
{'framework': 'pytorch or sklearn', 'task_type': 'regression', 'dataset_understanding': {'dataset_readme': '\nDATASET README:\n\n# DATASET README\n\n## DATASET OVERVIEW\n\n- Root

In [ ]:
#!pip install -q google-generativeai


In [31]:
!zip -r house_price_pred.zip /kaggle/working

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/.virtual_documents/ (stored 0%)
  adding: kaggle/working/.virtual_documents/__notebook_source__.ipynb (deflated 74%)
  adding: kaggle/working/submission.csv (deflated 54%)
  adding: kaggle/working/autonomous_ml/ (stored 0%)
  adding: kaggle/working/autonomous_ml/best_model/ (stored 0%)
  adding: kaggle/working/autonomous_ml/best_model/best_experiment/ (stored 0%)
  adding: kaggle/working/autonomous_ml/best_model/best_experiment/execution_log.txt (deflated 37%)
  adding: kaggle/working/autonomous_ml/best_model/best_experiment/training_plan.json (deflated 45%)
  adding: kaggle/working/autonomous_ml/best_model/best_experiment/generated_pipeline.py (deflated 52%)
  adding: kaggle/working/autonomous_ml/best_model/best_experiment/metrics.json (deflated 46%)
  adding: kaggle/working/autonomous_ml/best_model/best_metadata.json (deflated 71%)
  adding: kaggle/working/autonomous_ml/experiments/ (stored 0%)
  adding: kaggle/working/au

In [ ]:
!ls


In [ ]:
from IPython.display import FileLink
FileLink(r'house.zip')